# `parcelsim` — NYC Parcel Demand 2025 Projection

Generates synthetic last-mile parcel delivery demand for **New York City (5 boroughs)**
projected to **2025**, using ACS 2023 census data and updated e-commerce growth factors.

| | Validation notebook | This notebook |
|---|---|---|
| Purpose | Reproduce Yang et al. (2024) | Project demand to 2025 |
| Census year | ACS 2020 | ACS 2023 |
| Demand factor | 1.114 (2020→2021) | 1.45 (2020→2025) |
| Target parcels/day | ~1.91M (paper) | ~2.5M |

> **First run**: downloads ACS 2023 Census data + TIGER 2023 shapefiles (~25 MB).
> Cached separately from the 2020 data — `validation_nyc` is unaffected.

In [ ]:
!pip install -q "parcelsim[viz,us]"

In [ ]:
import os
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

import parcelsim
os.makedirs('outputs', exist_ok=True)
print(f'parcelsim {parcelsim.__version__}')

BOROUGH_FIPS = ['061', '047', '081', '005', '085']  # Manhattan, Brooklyn, Queens, Bronx, Staten Island
CRS = 'EPSG:32618'  # UTM Zone 18N

## 1. City

In [ ]:
from parcelsim.city import City

city = City.from_osmnx(
    query='New York City, New York, USA',
    crs=CRS,
    country_iso='US',
)
print(city)

## 2. Population — ACS 2023

ACS 5-year 2023 estimates (2019–2023), released December 2024.
Cached separately from the ACS 2020 data used in `validation_nyc.ipynb`.

In [ ]:
from parcelsim.population.adapters.census_us import USCensusAdapter

# OBTAIN A FREE API KEY AT https://api.census.gov/sign-up.html AND PUT IT HERE
CENSUS_API_KEY = "YOUR_CENSUS_API_KEY_HERE"

adapter = USCensusAdapter(
    state='NY',
    county_fips=BOROUGH_FIPS,
    acs_year=2023,
    cache_dir='parcelsim_cache',
    census_api_key=CENSUS_API_KEY,
)

population = adapter.build(city)

print(population.summary())
print(f'\nCensus tracts: {len(city.zones)}')
print(f'Total population:  {city.zones["population"].sum():>12,.0f}')
print(f'Total households:  {city.zones["n_households"].sum():>12,.0f}')

## 3. Parcel demand — 2025 projection

| Period | Factor | Source |
|---|---|---|
| 2020→2021 | 1.114 | Yang et al. (2024) / US Dept of Commerce |
| 2020→2025 | **1.45** | Calibrated to ~2.5M target — adjust if needed |

In [ ]:
from parcelsim.demand.usps_model import USPSDemandModel

model = USPSDemandModel(
    volume_increase_factor=1.45,   # 2020→2025 cumulative growth — adjust if needed
    usps_market_share=0.32,
    delivery_days_per_week=5,
)

demand = model.generate(population)
print(demand.summary())
print(f'\nTarget:  ~2,500,000 parcels/day')
print(f'Result:   {demand.total_delivery:>12,.0f} parcels/day')
print(f'Δ target: {(demand.total_delivery - 2_500_000) / 2_500_000 * 100:>+.1f}%')

## 4. Operator assignment & CA routing

In [ ]:
from parcelsim.operators.operator import OperatorRegistry
from parcelsim.operators.assignment import assign_parcels
from parcelsim.routing.ca.model import CARouter
from parcelsim.output.kpi import KPIReport

registry   = OperatorRegistry.from_builtin('us_2021')
assignment = assign_parcels(demand, registry, city)
result     = CARouter().solve(assignment, city)
report     = KPIReport.from_ca(result, assignment)

PAPER_2021 = {'delivery': 1_911_144, 'vkt': 98_871, 'trucks': 7_653}

w = 30
print(f"{'Metric':<{w}} {'2025 model':>12} {'2021 paper':>12} {'Δ%':>8}")
print('=' * (w + 36))
for name, mv, pv in [
    ('Delivery parcels/day', report.total_parcels_delivered, PAPER_2021['delivery']),
    ('VKT total (km/day)',   report.vkt_total_km,            PAPER_2021['vkt']),
    ('Trucks total',         report.n_trucks_total,          PAPER_2021['trucks']),
]:
    pct = (mv - pv) / pv * 100
    print(f"{name:<{w}} {mv:>12,.0f} {pv:>12,} {pct:>+8.1f}%")